# Exercise 2 — Industrial transfer learning with MVTec Capsule

This notebook is the more independent part of Exercise 2.

We use a small Hugging Face version of the **MVTec Capsule** category. This is an industrial visual inspection dataset derived from MVTec AD.

Important teaching caveat:

The original MVTec AD protocol is anomaly detection with defect-free training images.  
Here we create a simplified **supervised binary classification split** for teaching transfer learning:

- class 0: normal
- class 1: abnormal

This is useful for learning the pipeline, but it is not the official anomaly-detection benchmark protocol.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import torch

repo_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(repo_root / "src"))

from cvis_ml.config import DatasetConfig
from cvis_ml.data import MVTecCapsuleDataModule
from cvis_ml.models import TransferModelFactory, count_parameters, describe_trainable_parameters
from cvis_ml.engine import Trainer
from cvis_ml.visualization import show_batch, plot_history, show_confusion_matrix

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

## Task 1 — Configure the industrial dataset 🟢

The dataset is larger and more industrially relevant than CIFAR-10.

Keep the subset small for this exercise.

Recommended values:

- `image_size = 128`
- `batch_size = 16`
- `max_train_samples = 160`
- `max_val_samples = 80`

In [ ]:
# TODO:
# Complete the DatasetConfig.

mvtec_cfg = DatasetConfig(
    name="mvtec_capsule_teaching_split",
    data_root=str(repo_root / "data_cache"),
    image_size=...,          # TODO
    batch_size=...,          # TODO
    max_train_samples=...,   # TODO
    max_val_samples=...,     # TODO
    num_workers=2,
    seed=42,
)

mvtec_cfg

## Task 2 — Load the dataset 🟢 / 🟡

This may take some time on the first run because the dataset is downloaded from Hugging Face.

The data module:

1. loads MVTec Capsule,
2. creates a supervised teaching split,
3. resizes images,
4. applies ImageNet normalization,
5. returns PyTorch DataLoaders.

In [ ]:
# TODO:
# Create an MVTecCapsuleDataModule and call setup().

mvtec_dm = ...
mvtec_data = ...

print("Classes:", mvtec_data.class_names)
print("Number of classes:", mvtec_data.num_classes)
print("Train batches:", len(mvtec_data.train_loader))
print("Validation batches:", len(mvtec_data.val_loader))

show_batch(mvtec_data.train_loader, mvtec_data.class_names, n=8)

## Task 3 — Choose an industrial baseline 🟡

Choose a first baseline.

Recommended starting point:

- architecture: `mobilenet_v3_small`
- strategy: `frozen`

Why this is a good first baseline:

- efficient on CPU,
- low number of trainable parameters,
- useful for limited labeled data,
- good first diagnostic baseline.

In [ ]:
# TODO:
# Choose an architecture and transfer-learning strategy.

architecture = ...   # e.g. "mobilenet_v3_small" or "resnet18"
strategy = ...       # "frozen", "partial", or "full"

industrial_model = TransferModelFactory.create(
    architecture=architecture,
    num_classes=mvtec_data.num_classes,
    strategy=strategy,
    pretrained=True,
)

total, trainable = count_parameters(industrial_model)
print("Total parameters:", total)
print("Trainable parameters:", trainable)
for row in describe_trainable_parameters(industrial_model):
    print(row)

## Task 4 — Train the industrial baseline 🟡

Train for one epoch first.

Use a smaller learning rate if you choose `partial` or `full` fine-tuning.

In [ ]:
# TODO:
# Choose a learning rate based on the strategy.
# Hints:
# - frozen: 1e-3 is reasonable
# - partial/full: 1e-4 is safer

learning_rate = ...

industrial_trainer = Trainer(
    model=industrial_model,
    device="auto",
    learning_rate=learning_rate,
    weight_decay=1e-4,
)

industrial_result = industrial_trainer.fit(
    train_loader=mvtec_data.train_loader,
    val_loader=mvtec_data.val_loader,
    epochs=1,
    max_batches_per_epoch=None,
    name=f"{architecture}_{strategy}_mvtec_capsule",
)

plot_history(industrial_result.history, title="Industrial baseline")
show_confusion_matrix(industrial_result.y_true, industrial_result.y_pred, mvtec_data.class_names, title="Industrial baseline")

## Task 5 — Industrial interpretation 🟡

Answer in 6–10 sentences:

1. What validation accuracy and macro-F1 did you get?
2. Which class is more problematic?
3. Is accuracy alone sufficient here?
4. What could go wrong if normal/abnormal examples are not representative?
5. Was the augmentation operationally plausible?
6. Would you trust this as an industrial inspection model after one epoch? Why or why not?

In [ ]:
industrial_interpretation = """


"""
print(industrial_interpretation)

## Advanced task — Compare strategies 🔴

If you have time, compare:

1. MobileNetV3-Small frozen
2. ResNet18 frozen
3. ResNet18 partial fine-tuning

Record:

- validation accuracy,
- macro-F1,
- trainable parameter count,
- runtime,
- qualitative reliability.

In [ ]:
# Optional advanced comparison.
# You may copy the baseline loop from the CIFAR-10 notebook and adapt it here.